<a href="https://colab.research.google.com/github/marcelovolta/sr_uba_2026/blob/main/suprise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ! pip install "numpy<2" --upgrade
! pip uninstall numpy -y
! pip install numpy==1.26.4
## restart session

In [ ]:
import numpy as np
np.__version__

In [ ]:
! pip install --quiet scikit-surprise

In [ ]:
import surprise as sp
import pandas as pd
import sqlite3

In [ ]:
conn = sqlite3.connect('data.db')

df = pd.read_sql("""
    SELECT *
      FROM interacciones
     WHERE id_libro IN (SELECT id_libro FROM interacciones GROUP BY 1 HAVING count(*) > 20)
       AND id_lector IN (SELECT id_lector FROM interacciones GROUP BY 1 HAVING count(*) > 20)
     """, conn)

print(df.shape)
df.head()

In [ ]:
reader = sp.reader.Reader(rating_scale=(1, 10))

data = sp.dataset.Dataset.load_from_df(df[['id_lector', 'id_libro', 'rating']], reader) # cuidado con el orden
trainset, testset = sp.model_selection.train_test_split(data, test_size=0.2, random_state=42)

# Algoritmos de predicción

## BaselineOnly

In [ ]:
algo = sp.prediction_algorithms.baseline_only.BaselineOnly(verbose=False)
algo.fit(trainset)

predictions = algo.test(testset)

print(f"RMSE: { sp.accuracy.rmse(predictions) }")

## KNN

In [ ]:
sim_options = {
    'name': 'pearson', #'cosine', 'pearson_baseline'
    'user_based': True, #False,
}

algo = sp.prediction_algorithms.knns.KNNBasic(k=40, min_k=5, sim_options=sim_options, random_state=42)
algo.fit(trainset)
predictions = algo.test(testset)

print(f"RMSE: { sp.accuracy.rmse(predictions) }")

In [ ]:
sim_options = {
    'name': 'pearson', #'cosine', 'pearson_baseline'
    'user_based': True, #False,
}

algo = sp.prediction_algorithms.knns.KNNWithMeans(k=40, min_k=5, sim_options=sim_options, random_state=42)
algo.fit(trainset)
predictions = algo.test(testset)

print(f"RMSE: { sp.accuracy.rmse(predictions) }")

## SVD

In [ ]:
algo = sp.prediction_algorithms.matrix_factorization.SVD(n_factors=500, n_epochs=20, random_state=42)

algo.fit(trainset)
predictions = algo.test(testset)

print(f"RMSE: { sp.accuracy.rmse(predictions) }")

## NMF

In [ ]:
algo = sp.prediction_algorithms.matrix_factorization.NMF(n_factors=500, n_epochs=20, random_state=42)

algo.fit(trainset)
predictions = algo.test(testset)

print(f"RMSE: { sp.accuracy.rmse(predictions) }")

# Predicción

In [ ]:
%%time
conn = sqlite3.connect("data.db")
df = pd.read_sql('SELECT * FROM interacciones;', conn)

reader = sp.reader.Reader(rating_scale=(1, 10))
data = sp.dataset.Dataset.load_from_df(df[['id_lector', 'id_libro', 'rating']], reader) # cuidado con el orden

In [ ]:
%%time

# algoritmo
algo = sp.prediction_algorithms.baseline_only.BaselineOnly(verbose=False)
algo.fit(data.build_full_trainset())

In [ ]:
sp.dump.dump("algo.pkl", algo=algo)
del algo

In [ ]:
_, algo = sp.dump.load("algo.pkl")

In [ ]:
lector_id = "popocito"

libro_id = "reina-roja"
print(libro_id, algo.predict(lector_id, libro_id, 9))
print(libro_id, algo.predict(lector_id, libro_id, 9).est)

libro_id = "la-novia-rebelde"
print(libro_id, algo.predict(lector_id, libro_id, 7).est)